In [9]:
import pandas as pd
import requests
import json
import pandas as pd
from tqdm import tqdm

In [10]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:latest"

In [11]:

df = pd.read_csv("mozilla_core_clean.csv")
# Rename by specific mapping
df = df.rename(columns={"Unnamed: 0": "id"})
eval_docs = df.sample(400, random_state=42)

In [12]:
eval_docs.head()

,id,Component,Title,Description,Created_time,Resolved_time
168701,170060,Layout: Tables,teste,User Agent: Mozilla/5.0 (Windows NT 5.1; rv:8....,2012-01-06 05:14:27 -0800,2012-01-06 05:45:03 -0800
156712,157698,Graphics,[Linux] With layers acceleration; doorhangers ...,Created attachment 508533; screenshot of bug; ...,2011-01-31 13:58:18 -0800,2012-09-24 17:57:23 -0700
172952,174483,WebRTC: Signaling,Various run-only-one-instance bugs in media/we...,There are several instances of using msg queue...,2012-04-13 14:58:20 -0700,2012-10-04 02:54:35 -0700
30626,30847,Networking: Cache,javascript rarely renders rollovers in M18 (20...,in m18 goto the above address and try to use t...,2000-11-02 22:53:29 -0800,2001-03-01 19:28:26 -0800
169650,171044,Editor,Various nsEditor cleanup,Created attachment 592809; Patch v1,2012-01-30 11:58:37 -0800,2012-02-01 05:46:57 -0800


In [13]:
eval_docs.to_csv("random_samples.csv", index=False)

In [27]:
def generate_queries(bug_text):
    prompt = f"""
You are generating evaluation queries for a retrieval system.

Given the following bug report, generate EXACTLY 2 realistic developer search queries.

Return output strictly in JSON format like this:
{{
  "queries": [
    "query 1",
    "query 2"
  ]
}}

BUG REPORT:
\"\"\"{bug_text}\"\"\"
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3
        }
    }

    response = requests.post(OLLAMA_URL, json=payload)
    result = response.json()["response"]
    # result = None
    # print(response.status_code)
    # print(response.text)
    # Extract JSON safely
    try:
        json_start = result.find("{")
        json_end = result.rfind("}") + 1
        parsed = json.loads(result[json_start:json_end])
        return parsed["queries"]
    except:
        return []

In [28]:
evaluation_data = []

for _, row in tqdm(eval_docs.iterrows(), total=len(eval_docs)):
    bug_text = row["Description"]
    doc_id = row["id"]

    queries = generate_queries(bug_text)

    for q in queries:
        evaluation_data.append({
            "query": q,
            "relevant_doc_id": doc_id
        })

eval_df = pd.DataFrame(evaluation_data)
eval_df.to_csv("evaluation_queries.csv", index=False)

print("Saved evaluation_queries.csv")

100%|██████████| 5/5 [01:52<00:00, 22.60s/it]

Saved evaluation_queries.csv
